
# Adaptive SVD-Based Low-Rank Compression for Efficient CCTV Video Storage

This Google Colab notebook implements:

- CCTV video upload and frame extraction
- SVD-based low-rank image compression
- Dynamic rank selection
- MSE, PSNR and SSIM
- Compression ratio and storage savings
- 40–70% storage-reduction analysis
- 3× storage-efficiency analysis
- 24-hour CCTV storage simulation
- Storage-capacity → hours/days estimation
- Compression vs. quality dashboard
- Compression/reconstruction time and FPS benchmark
- JPEG comparison
- Reconstructed CCTV video
- CSV result export
- Automated final project report

> **Note:** SVD is demonstrated on grayscale frames. The SVD representation size is calculated mathematically; an MP4 file created from reconstructed frames is a separate video-encoding step.


In [ ]:
!pip install -q opencv-python scikit-image pandas matplotlib seaborn

In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time

from skimage.metrics import structural_similarity as ssim
from IPython.display import display, Video
from google.colab import files


## 1. Upload CCTV video

In [ ]:

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

print("Uploaded video:", video_path)


In [ ]:

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frame_count / fps if fps > 0 else 0

print("Video Information")
print("------------------")
print("Resolution :", width, "x", height)
print("FPS        :", fps)
print("Frames     :", frame_count)
print("Duration   :", round(duration, 2), "seconds")
print("Duration   :", round(duration / 60, 2), "minutes")

cap.release()


## 2. Display an original CCTV frame

In [ ]:

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 7))
    plt.imshow(frame_rgb)
    plt.title("Original CCTV Frame")
    plt.axis("off")
    plt.show()

cap.release()


## 3. SVD compression function

In [ ]:

def svd_compress(image, k):
    """
    Compress a grayscale image using rank-k SVD.

    A ≈ U_k Sigma_k V_k^T

    Returns:
        reconstructed image
        compression ratio
        storage saved percentage
    """

    image = image.astype(np.float32)

    U, S, Vt = np.linalg.svd(image, full_matrices=False)

    k = min(k, len(S))

    Uk = U[:, :k]
    Sk = S[:k]
    Vtk = Vt[:k, :]

    reconstructed = Uk @ np.diag(Sk) @ Vtk
    reconstructed = np.clip(reconstructed, 0, 255).astype(np.uint8)

    # Original representation: H*W values
    original_storage = image.shape[0] * image.shape[1]

    # Rank-k SVD representation:
    # U = H*k, Sigma = k, V^T = k*W
    compressed_storage = (
        image.shape[0] * k +
        k +
        k * image.shape[1]
    )

    compression_ratio = original_storage / compressed_storage

    storage_saved = (
        1 - compressed_storage / original_storage
    ) * 100

    return reconstructed, compression_ratio, storage_saved


## 4. Test SVD compression on one frame

In [ ]:

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

k = 120

compressed_frame, ratio, saved = svd_compress(gray, k)

print("SVD Compression")
print("----------------")
print("Rank k            :", k)
print("Compression ratio :", round(ratio, 2), "x")
print("Storage saved     :", round(saved, 2), "%")


In [ ]:

plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.imshow(gray, cmap="gray")
plt.title("Original Grayscale")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(compressed_frame, cmap="gray")
plt.title(f"SVD Compressed (k={k})")
plt.axis("off")

plt.show()


## 5. MSE, PSNR and SSIM

In [ ]:

def calculate_metrics(original, reconstructed):
    original = original.astype(np.float32)
    reconstructed = reconstructed.astype(np.float32)

    mse = np.mean((original - reconstructed) ** 2)

    if mse == 0:
        psnr_value = float("inf")
    else:
        psnr_value = 10 * np.log10((255 ** 2) / mse)

    ssim_value = ssim(
        original.astype(np.uint8),
        reconstructed.astype(np.uint8),
        data_range=255
    )

    return mse, psnr_value, ssim_value


In [ ]:

mse, psnr_value, ssim_value = calculate_metrics(
    gray,
    compressed_frame
)

print("Quality Metrics")
print("----------------")
print("MSE  :", round(mse, 4))
print("PSNR :", round(psnr_value, 2), "dB")
print("SSIM :", round(ssim_value, 4))


## 6. Evaluate multiple SVD ranks

In [ ]:

ranks = [20, 30, 50, 75, 100, 120, 150]

# Do not test ranks larger than the frame's smaller dimension.
ranks = [r for r in ranks if r <= min(gray.shape)]

results = []

for k in ranks:
    start_time = time.time()

    reconstructed, ratio, saved = svd_compress(gray, k)

    processing_time = time.time() - start_time

    mse, psnr_value, ssim_value = calculate_metrics(
        gray,
        reconstructed
    )

    results.append({
        "Rank": k,
        "Compression Ratio": ratio,
        "Storage Saved (%)": saved,
        "MSE": mse,
        "PSNR (dB)": psnr_value,
        "SSIM": ssim_value,
        "Time (sec)": processing_time
    })

results_df = pd.DataFrame(results)

display(results_df)


## 7. PSNR vs. rank

In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    results_df["Rank"],
    results_df["PSNR (dB)"],
    marker="o"
)

plt.xlabel("SVD Rank (k)")
plt.ylabel("PSNR (dB)")
plt.title("PSNR vs SVD Rank")
plt.grid(True)
plt.show()


## 8. SSIM vs. rank

In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    results_df["Rank"],
    results_df["SSIM"],
    marker="o"
)

plt.xlabel("SVD Rank (k)")
plt.ylabel("SSIM")
plt.title("SSIM vs SVD Rank")
plt.grid(True)
plt.show()


## 9. Storage savings vs. rank

In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    results_df["Rank"],
    results_df["Storage Saved (%)"],
    marker="o"
)

plt.xlabel("SVD Rank (k)")
plt.ylabel("Storage Saved (%)")
plt.title("Storage Savings vs SVD Rank")
plt.grid(True)
plt.show()


## 10. Compression ratio vs. quality

In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    results_df["Compression Ratio"],
    results_df["PSNR (dB)"],
    marker="o"
)

plt.xlabel("Compression Ratio")
plt.ylabel("PSNR (dB)")
plt.title("Compression Ratio vs Image Quality")
plt.grid(True)
plt.show()


## 11. Minimum rank for a target PSNR

In [ ]:

target_psnr = 30

valid_results = results_df[
    results_df["PSNR (dB)"] >= target_psnr
]

if len(valid_results) > 0:
    minimum_rank = int(valid_results.iloc[0]["Rank"])

    print("Target PSNR:", target_psnr, "dB")
    print("Minimum required rank:", minimum_rank)
else:
    minimum_rank = None
    print("No tested rank achieved the target PSNR.")


## 12. Automatic quality-based rank selection

In [ ]:

def select_rank_for_quality(results_df, target_psnr=30, target_ssim=None):
    candidates = results_df[
        results_df["PSNR (dB)"] >= target_psnr
    ].copy()

    if target_ssim is not None:
        candidates = candidates[
            candidates["SSIM"] >= target_ssim
        ]

    if len(candidates) == 0:
        return None

    # Smallest rank that meets the quality requirement
    return int(candidates.iloc[0]["Rank"])


selected_rank = select_rank_for_quality(
    results_df,
    target_psnr=30
)

print("Automatically selected rank:", selected_rank)


## 13. Dynamic compression based on storage requirement

In [ ]:

def dynamic_rank(results_df, required_storage_saving):
    candidates = results_df[
        results_df["Storage Saved (%)"] >= required_storage_saving
    ].copy()

    if len(candidates) == 0:
        return None

    # Select the largest rank that still satisfies the storage requirement,
    # preserving as much quality as possible.
    selected = candidates.iloc[-1]

    return int(selected["Rank"])


required_saving = 50

dynamic_k = dynamic_rank(
    results_df,
    required_saving
)

print("Required storage saving:", required_saving, "%")
print("Selected SVD rank:", dynamic_k)


## 14. Visual comparison of compression levels

In [ ]:

comparison_ranks = [r for r in [10, 30, 50, 100] if r <= min(gray.shape)]

plt.figure(figsize=(16, 10))

for i, k in enumerate(comparison_ranks):
    reconstructed, _, _ = svd_compress(gray, k)

    plt.subplot(2, 2, i + 1)
    plt.imshow(reconstructed, cmap="gray")
    plt.title(f"SVD Rank = {k}")
    plt.axis("off")

plt.tight_layout()
plt.show()


## 15. Extract representative CCTV frames

In [ ]:

def extract_frames(video_path, number_of_frames=20):
    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frame_indices = np.linspace(
        0,
        max(total_frames - 1, 0),
        number_of_frames,
        dtype=int
    )

    frames = []

    for index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))

        ret, current_frame = cap.read()

        if ret:
            gray_frame = cv2.cvtColor(
                current_frame,
                cv2.COLOR_BGR2GRAY
            )
            frames.append(gray_frame)

    cap.release()

    return frames


frames = extract_frames(
    video_path,
    number_of_frames=20
)

print("Frames extracted:", len(frames))


## 16. Evaluate SVD across multiple frames

In [ ]:

test_rank = 120
test_rank = min(test_rank, min(frames[0].shape))

frame_results = []

for i, current_frame in enumerate(frames):
    start = time.time()

    reconstructed, ratio, saved = svd_compress(
        current_frame,
        test_rank
    )

    processing_time = time.time() - start

    mse, psnr_value, ssim_value = calculate_metrics(
        current_frame,
        reconstructed
    )

    frame_results.append({
        "Frame": i + 1,
        "Rank": test_rank,
        "MSE": mse,
        "PSNR": psnr_value,
        "SSIM": ssim_value,
        "Compression Ratio": ratio,
        "Storage Saved (%)": saved,
        "Processing Time": processing_time
    })

frame_results_df = pd.DataFrame(frame_results)

display(frame_results_df)


## 17. Average multi-frame performance

In [ ]:

print("Average Performance")
print("-------------------")

print(
    "Average MSE:",
    round(frame_results_df["MSE"].mean(), 3)
)

print(
    "Average PSNR:",
    round(frame_results_df["PSNR"].mean(), 2),
    "dB"
)

print(
    "Average SSIM:",
    round(frame_results_df["SSIM"].mean(), 4)
)

print(
    "Average Storage Saved:",
    round(frame_results_df["Storage Saved (%)"].mean(), 2),
    "%"
)

print(
    "Average Compression Ratio:",
    round(frame_results_df["Compression Ratio"].mean(), 2),
    "x"
)


## 18. Compression/reconstruction speed benchmark

In [ ]:

benchmark_frames = frames[:10]

times = []

for current_frame in benchmark_frames:
    start = time.time()

    reconstructed, _, _ = svd_compress(
        current_frame,
        test_rank
    )

    end = time.time()
    times.append(end - start)

average_time = np.mean(times)
processing_fps = 1 / average_time if average_time > 0 else 0

print("Performance Benchmark")
print("---------------------")
print("Average processing time:",
      round(average_time, 4), "seconds/frame")

print("Processing rate:",
      round(processing_fps, 2), "FPS")


## 19. Check real-time capability

In [ ]:

print("CCTV FPS:", round(fps, 2))
print("SVD processing FPS:", round(processing_fps, 2))

if processing_fps >= fps:
    print("Result: The tested configuration can keep up with the video FPS.")
else:
    print("Result: The tested configuration is slower than the source video FPS.")


## 20. Storage calculation functions

In [ ]:

def storage_for_video(width, height, fps, hours, channels=1):
    bytes_per_frame = width * height * channels
    total_frames = fps * 3600 * hours
    total_bytes = bytes_per_frame * total_frames
    return total_bytes


def bytes_to_gb(bytes_value):
    return bytes_value / (1024 ** 3)


def storage_duration(
    storage_capacity_gb,
    width,
    height,
    fps,
    compression_ratio,
    channels=1
):
    bytes_per_frame = width * height * channels

    seconds_available = (
        storage_capacity_gb *
        (1024 ** 3) *
        compression_ratio
    ) / (bytes_per_frame * fps)

    hours = seconds_available / 3600
    days = hours / 24

    return hours, days


## 21. 24-hour CCTV storage simulation

In [ ]:

hours = 24

original_storage = storage_for_video(
    width,
    height,
    fps,
    hours,
    channels=1
)

print("24-hour CCTV simulation")
print("------------------------")
print(
    "Uncompressed grayscale representation:",
    round(bytes_to_gb(original_storage), 2),
    "GB"
)


## 22. Calculate compressed SVD storage

In [ ]:

test_row = results_df[
    results_df["Rank"] == test_rank
]

if len(test_row) == 0:
    # If rank 50 was not in the tested list, use the closest available rank.
    closest_rank = min(
        results_df["Rank"],
        key=lambda r: abs(r - test_rank)
    )
    test_rank = int(closest_rank)
    test_row = results_df[results_df["Rank"] == test_rank]

compressed_ratio = float(
    test_row["Compression Ratio"].iloc[0]
)

compressed_storage = original_storage / compressed_ratio

print("Using SVD rank:", test_rank)
print("Original storage:",
      round(bytes_to_gb(original_storage), 2),
      "GB")

print("SVD representation storage:",
      round(bytes_to_gb(compressed_storage), 2),
      "GB")

print("Storage saved:",
      round(
          (1 - compressed_storage / original_storage) * 100,
          2
      ),
      "%")


## 23. Storage capacity → CCTV hours/days

In [ ]:

storage_capacity = 500  # GB

hours_available, days_available = storage_duration(
    storage_capacity,
    width,
    height,
    fps,
    compressed_ratio,
    channels=1
)

print("Available storage:", storage_capacity, "GB")
print("Estimated CCTV storage time:")
print("Hours:", round(hours_available, 2))
print("Days :", round(days_available, 2))


## 24. 24-hour storage comparison graph

In [ ]:

storage_data = pd.DataFrame({
    "Method": ["Uncompressed", "SVD"],
    "Storage (GB)": [
        bytes_to_gb(original_storage),
        bytes_to_gb(compressed_storage)
    ]
})

plt.figure(figsize=(8, 6))

sns.barplot(
    data=storage_data,
    x="Method",
    y="Storage (GB)"
)

plt.title("24-Hour CCTV Storage Comparison")
plt.ylabel("Storage Required (GB)")
plt.show()


## 25. JPEG compression function

In [ ]:

def jpeg_compress(image, quality=50):
    encode_param = [
        int(cv2.IMWRITE_JPEG_QUALITY),
        quality
    ]

    success, encoded = cv2.imencode(
        ".jpg",
        image,
        encode_param
    )

    if not success:
        return None, None, None

    reconstructed = cv2.imdecode(
        encoded,
        cv2.IMREAD_GRAYSCALE
    )

    compressed_size = len(encoded)
    original_size = image.shape[0] * image.shape[1]

    ratio = original_size / compressed_size
    saved = (1 - compressed_size / original_size) * 100

    return reconstructed, ratio, saved


## 26. JPEG comparison

In [ ]:

jpeg_quality = 50

jpeg_frame, jpeg_ratio, jpeg_saved = jpeg_compress(
    gray,
    jpeg_quality
)

jpeg_mse, jpeg_psnr, jpeg_ssim = calculate_metrics(
    gray,
    jpeg_frame
)

print("JPEG Results")
print("------------")
print("Quality setting   :", jpeg_quality)
print("Compression ratio :", round(jpeg_ratio, 2), "x")
print("Storage saved     :", round(jpeg_saved, 2), "%")
print("MSE               :", round(jpeg_mse, 3))
print("PSNR              :", round(jpeg_psnr, 2), "dB")
print("SSIM              :", round(jpeg_ssim, 4))


## 27. Visual SVD vs JPEG comparison

In [ ]:

svd_frame, svd_ratio, svd_saved = svd_compress(
    gray,
    test_rank
)

svd_mse, svd_psnr, svd_ssim = calculate_metrics(
    gray,
    svd_frame
)

plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(svd_frame, cmap="gray")
plt.title(
    f"SVD k={test_rank}\n"
    f"PSNR={svd_psnr:.2f} dB, SSIM={svd_ssim:.3f}"
)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(jpeg_frame, cmap="gray")
plt.title(
    f"JPEG Q={jpeg_quality}\n"
    f"PSNR={jpeg_psnr:.2f} dB, SSIM={jpeg_ssim:.3f}"
)
plt.axis("off")

plt.tight_layout()
plt.show()


## 28. SVD vs JPEG quantitative table

In [ ]:

comparison = pd.DataFrame({
    "Method": ["SVD", "JPEG"],
    "Parameter": [f"k={test_rank}", f"Quality={jpeg_quality}"],
    "Compression Ratio": [svd_ratio, jpeg_ratio],
    "Storage Saved (%)": [svd_saved, jpeg_saved],
    "MSE": [svd_mse, jpeg_mse],
    "PSNR (dB)": [svd_psnr, jpeg_psnr],
    "SSIM": [svd_ssim, jpeg_ssim]
})

display(comparison)


## 29. Compression vs. quality dashboard

In [ ]:

fig = plt.figure(figsize=(16, 10))

ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(
    results_df["Rank"],
    results_df["PSNR (dB)"],
    marker="o"
)
ax1.set_title("Rank vs PSNR")
ax1.set_xlabel("Rank")
ax1.set_ylabel("PSNR (dB)")
ax1.grid(True)

ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(
    results_df["Rank"],
    results_df["SSIM"],
    marker="o"
)
ax2.set_title("Rank vs SSIM")
ax2.set_xlabel("Rank")
ax2.set_ylabel("SSIM")
ax2.grid(True)

ax3 = fig.add_subplot(2, 2, 3)
ax3.plot(
    results_df["Rank"],
    results_df["Storage Saved (%)"],
    marker="o"
)
ax3.set_title("Rank vs Storage Saved")
ax3.set_xlabel("Rank")
ax3.set_ylabel("Storage Saved (%)")
ax3.grid(True)

ax4 = fig.add_subplot(2, 2, 4)
ax4.plot(
    results_df["Rank"],
    results_df["Compression Ratio"],
    marker="o"
)
ax4.set_title("Rank vs Compression Ratio")
ax4.set_xlabel("Rank")
ax4.set_ylabel("Compression Ratio")
ax4.grid(True)

plt.tight_layout()
plt.show()


## 30. Check the 40–70% storage-reduction target

In [ ]:

target_min = 40
target_max = 70

target_results = results_df[
    (results_df["Storage Saved (%)"] >= target_min) &
    (results_df["Storage Saved (%)"] <= target_max)
]

print("Ranks achieving 40–70% storage reduction:")
display(target_results)


## 31. Check the 3× storage-efficiency requirement

In [ ]:

required_ratio = 3

qualified = results_df[
    results_df["Compression Ratio"] >= required_ratio
]

print("Ranks achieving at least 3× storage efficiency:")
display(qualified)


## 32. Generate reconstructed CCTV video

In [ ]:

input_video = video_path
output_video = "svd_reconstructed_cctv.mp4"

cap = cv2.VideoCapture(input_video)

video_fps = cap.get(cv2.CAP_PROP_FPS)
original_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
original_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_video,
    fourcc,
    video_fps,
    (original_width, original_height),
    False
)

# Increase this for a longer demonstration.
# SVD is computationally expensive, so start with 100 frames.
max_frames = 100
count = 0

start_total = time.time()

while cap.isOpened() and count < max_frames:
    ret, current_frame = cap.read()

    if not ret:
        break

    gray_current = cv2.cvtColor(
        current_frame,
        cv2.COLOR_BGR2GRAY
    )

    reconstructed, _, _ = svd_compress(
        gray_current,
        test_rank
    )

    out.write(reconstructed)
    count += 1

cap.release()
out.release()

total_time = time.time() - start_total

print("Reconstructed frames:", count)
print("Total processing time:", round(total_time, 2), "seconds")
print("Processing FPS:", round(count / total_time, 2) if total_time > 0 else 0)
print("Output:", output_video)


## 33. Play reconstructed CCTV video

In [ ]:
Video(output_video, embed=True)


In [ ]:
files.download('svd_reconstructed_cctv.mp4')

## 34. Export results to CSV

In [ ]:

results_df.to_csv(
    "svd_compression_results.csv",
    index=False
)

frame_results_df.to_csv(
    "frame_level_results.csv",
    index=False
)

comparison.to_csv(
    "svd_vs_jpeg.csv",
    index=False
)

print("CSV files created successfully.")


## 35. Download result files

In [ ]:

files.download("svd_compression_results.csv")


In [ ]:

files.download("frame_level_results.csv")


In [ ]:

files.download("svd_vs_jpeg.csv")


## 36. Automated final project report

In [ ]:

print("=" * 65)
print("       CCTV SVD COMPRESSION PROJECT REPORT")
print("=" * 65)

print("\nVIDEO")
print("-" * 65)
print("Resolution :", width, "x", height)
print("FPS        :", round(fps, 2))
print("Frames     :", frame_count)
print("Duration   :", round(duration, 2), "seconds")

print("\nSVD COMPRESSION")
print("-" * 65)
print("Test Rank          :", test_rank)
print("Compression Ratio  :", round(svd_ratio, 2), "x")
print("Storage Saved      :", round(svd_saved, 2), "%")

print("\nIMAGE QUALITY")
print("-" * 65)
print("MSE                :", round(svd_mse, 3))
print("PSNR               :", round(svd_psnr, 2), "dB")
print("SSIM               :", round(svd_ssim, 4))

print("\nPERFORMANCE")
print("-" * 65)
print("SVD Processing FPS :", round(processing_fps, 2))
print("Source Video FPS   :", round(fps, 2))

print("\n24-HOUR STORAGE")
print("-" * 65)
print("Original           :", round(bytes_to_gb(original_storage), 2), "GB")
print("SVD Representation :", round(bytes_to_gb(compressed_storage), 2), "GB")
print("Storage Saved      :",
      round((1 - compressed_storage / original_storage) * 100, 2), "%")

print("\nSTORAGE CAPACITY")
print("-" * 65)
print("Available Storage  :", storage_capacity, "GB")
print("Estimated Hours    :", round(hours_available, 2))
print("Estimated Days     :", round(days_available, 2))

print("\nJPEG COMPARISON")
print("-" * 65)
print("JPEG PSNR          :", round(jpeg_psnr, 2), "dB")
print("JPEG SSIM          :", round(jpeg_ssim, 4))
print("JPEG Storage Saved :", round(jpeg_saved, 2), "%")

print("\nQUALITY-BASED RANK")
print("-" * 65)
print("Target PSNR        :", target_psnr, "dB")
print("Minimum Rank       :", minimum_rank)

print("\n" + "=" * 65)
print("                    END OF REPORT")
print("=" * 65)
